# ML-Based Prediction of Polymer Glass Transition Temperature (Tg)

**Dhruv Jariwala**  
Portfolio implementation of a CL653 *Applications of AI and ML for Chemical Engineering* project.

This notebook predicts polymer glass transition temperature (**Tg**) from SMILES-derived structural features and molecular descriptors and compares Linear Regression, Random Forest, and XGBoost.


## 1. Workflow

1. Load and clean polymer records containing SMILES, polymer class, and Tg.
2. Generate simple structural features and RDKit molecular descriptors.
3. Use a grouped train-test split by SMILES to avoid identical structures appearing in both sets.
4. One-hot encode polymer class; scale numeric features for Linear Regression.
5. Train Dummy, Linear Regression, Random Forest, and XGBoost models.
6. Evaluate with MAE, RMSE, R², parity plots, residuals, and feature importance.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBRegressor

RANDOM_STATE = 42


## 2. Load and clean data

Download the Kaggle dataset linked in the repository README and place `TgSS_enriched_cleaned.csv` inside the `data/` folder.


In [ ]:
DATA_CANDIDATES = [Path("data/TgSS_enriched_cleaned.csv"), Path("../data/TgSS_enriched_cleaned.csv")]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place TgSS_enriched_cleaned.csv inside the data/ folder.")

df_raw = pd.read_csv(DATA_PATH)
df = (df_raw[["SMILES", "Polymer Class", "Tg"]]
      .dropna()
      .drop_duplicates()
      .reset_index(drop=True))
print("Rows:", len(df), "| Classes:", df["Polymer Class"].nunique())


## 3. Molecular descriptors


In [ ]:
def descriptor_row(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return {
        "smiles_length": len(smiles),
        "num_C": smiles.count("C"),
        "num_O": smiles.count("O"),
        "num_N": smiles.count("N"),
        "MolWt": Descriptors.MolWt(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "RotatableBonds": Descriptors.NumRotatableBonds(mol),
        "RingCount": rdMolDescriptors.CalcNumRings(mol),
        "HDonors": rdMolDescriptors.CalcNumHBD(mol),
        "HAcceptors": rdMolDescriptors.CalcNumHBA(mol),
        "AromaticRings": rdMolDescriptors.CalcNumAromaticRings(mol),
    }

desc = pd.DataFrame([descriptor_row(s) for s in df["SMILES"]], index=df.index)
valid = desc.notna().all(axis=1)
df = pd.concat([df.loc[valid].reset_index(drop=True), desc.loc[valid].reset_index(drop=True)], axis=1)
print("Final modeling shape:", df.shape)


## 4. Grouped train-test split


In [ ]:
feature_cols = [
    "Polymer Class", "smiles_length", "num_C", "num_O", "num_N", "MolWt", "TPSA",
    "RotatableBonds", "RingCount", "HDonors", "HAcceptors", "AromaticRings"
]
X = df[feature_cols]
y = df["Tg"]
groups = df["SMILES"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train:", len(train_idx), "| Test:", len(test_idx))
print("SMILES overlap:", len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))


## 5. Preprocessing


In [ ]:
categorical = ["Polymer Class"]
numeric = [c for c in feature_cols if c not in categorical]

linear_preprocessor = ColumnTransformer([
    ("class", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("numeric", StandardScaler(), numeric),
])
tree_preprocessor = ColumnTransformer([
    ("class", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
], remainder="passthrough")

X_train_linear = linear_preprocessor.fit_transform(X_train)
X_test_linear = linear_preprocessor.transform(X_test)
X_train_tree = tree_preprocessor.fit_transform(X_train)
X_test_tree = tree_preprocessor.transform(X_test)


## 6. Train models


In [ ]:
dummy = DummyRegressor(strategy="mean").fit(X_train_tree, y_train)
linear = LinearRegression().fit(X_train_linear, y_train)
random_forest = RandomForestRegressor(
    n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1
).fit(X_train_tree, y_train)
xgboost = XGBRegressor(
    n_estimators=700, learning_rate=0.05, max_depth=6,
    subsample=0.85, colsample_bytree=0.85,
    random_state=RANDOM_STATE, objective="reg:squarederror"
).fit(X_train_tree, y_train)


## 7. Evaluate models


In [ ]:
def metrics(name, model, Xtr, Xte):
    p_tr, p_te = model.predict(Xtr), model.predict(Xte)
    return {
        "Model": name,
        "Train_MAE": mean_absolute_error(y_train, p_tr),
        "Train_RMSE": np.sqrt(mean_squared_error(y_train, p_tr)),
        "Train_R2": r2_score(y_train, p_tr),
        "Test_MAE": mean_absolute_error(y_test, p_te),
        "Test_RMSE": np.sqrt(mean_squared_error(y_test, p_te)),
        "Test_R2": r2_score(y_test, p_te),
    }

results = pd.DataFrame([
    metrics("Dummy", dummy, X_train_tree, X_test_tree),
    metrics("Linear Regression", linear, X_train_linear, X_test_linear),
    metrics("Random Forest", random_forest, X_train_tree, X_test_tree),
    metrics("XGBoost", xgboost, X_train_tree, X_test_tree),
])
results.round(4)


In [ ]:
comparison = results.sort_values("Test_RMSE")
plt.figure(figsize=(8, 4.5))
plt.bar(comparison["Model"], comparison["Test_RMSE"])
plt.ylabel("Test RMSE (°C)")
plt.title("Model comparison")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 8. Feature importance


In [ ]:
feature_names = tree_preprocessor.get_feature_names_out()
importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": random_forest.feature_importances_,
}).sort_values("Importance", ascending=False)

importance["Feature"] = (importance["Feature"]
    .str.replace("remainder__", "", regex=False)
    .str.replace("class__", "", regex=False))
importance.head(15)


## 9. Best-model diagnostics


In [ ]:
best_predictions = xgboost.predict(X_test_tree)

plt.figure(figsize=(6, 6))
plt.scatter(y_test, best_predictions, alpha=0.45)
lims = [min(y_test.min(), best_predictions.min()), max(y_test.max(), best_predictions.max())]
plt.plot(lims, lims, "--")
plt.xlabel("Actual Tg (°C)")
plt.ylabel("Predicted Tg (°C)")
plt.title("XGBoost: actual vs predicted Tg")
plt.tight_layout()
plt.show()

residuals = y_test - best_predictions
plt.figure(figsize=(7, 4.5))
plt.scatter(best_predictions, residuals, alpha=0.45)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Tg (°C)")
plt.ylabel("Residual (°C)")
plt.title("XGBoost residual plot")
plt.tight_layout()
plt.show()


## 10. Conclusion

- Non-linear ensemble models outperform Linear Regression on this dataset.
- Grouping the split by SMILES prevents direct overlap of identical polymer structures between train and test sets.
- This remains a structure-based baseline; actual Tg also depends on formulation, processing history, molecular-weight distribution, curing conditions, and other variables not available here.
- Future work can include repeated grouped cross-validation, hyperparameter optimization, uncertainty estimation, and richer molecular representations.
